<a href="https://colab.research.google.com/github/HC-AIGroup/CS5330_F26/blob/main/Module%205/Lab_Studio_3_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Lab Studio 3 · Train it, break it, fix it
### CS 5330 · Module 5: Deep Learning & CNN Foundations

**Work in pairs · Google Colab with a GPU** (*Runtime → Change runtime type → T4 GPU*) · about **2 hours**, five missions

In the lecture you saw how CNNs work. Today you'll find out what it **feels** like to train one: you'll make it work, watch it break, and fix it.

| Mission | Time | What you'll do |
|---|---|---|
| 1 · Build a CNN | 25 min | Predict every shape, write the training step, beat your Lab 4 score |
| 2 · Training doctor | 25 min | Four broken training runs. Find the bug from evidence, then fix it |
| 3 · See what it sees | 15 min | Look inside a real ImageNet network, using your own face |
| 4 · Teach it your objects | 25 min | Transfer learning on photos you record with your webcam |
| 5 · YOLO hunt | 15 min | Make a real detector succeed, fail and hallucinate |
| 🏁 Speed-run + log | 10 min | Leaderboard and a 3-question engineering log |

## 📜 Studio rules: read this first

This lab is **not graded on code**. Most of the code is already written. You're graded on three things:

1. **🔮 Predictions.** Before many cells you write down a guess. Guess honestly; wrong guesses cost nothing. At the end, your **intuition score** shows how your gut feeling improved. Looking up the answer first only fools yourself.
2. **🎤 TA checkpoints (3).** You explain something out loud to a TA, and the TA gives you a code word to unlock the next part. You can't copy a conversation.
3. **📝 Engineering log.** Three short answers in your own words, at the end.

**Please keep AI assistants closed today.** Not because it's cheating, but because this lab's whole point is the "aha" moment, and an AI would take that from you. If you're stuck, ask your partner first, then a TA.

**About the blanks:** `___` marks a blank you fill in. If you see `NameError: name '___' is not defined`, you missed one.

---
## Part 0 · Setup (3 min)
Run both cells. The second one downloads CIFAR-10, which takes about 30 seconds. We use the **same 4 classes as Lab 4** so you can compare with your HOG + SVM result.

In [ ]:
import time, math, copy, base64, json, hashlib, random
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from IPython.display import display, clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0); random.seed(0)
plt.rcParams["figure.dpi"] = 90
print("Running on:", device.upper(), "✅" if device == "cuda" else "(slower, but fine. Runtime → Change runtime type → T4 GPU if you can)")

# ---------- prediction tracker (your intuition score)
PRED = {}
def check(name, guess, actual, tol=0.0):
    """Compare a guess with the real value and remember the result for your intuition score."""
    if guess is None or (isinstance(guess, str) and guess.strip() in ("", "___")):
        print(f"🔮 {name}: no guess written. Write one first; that's the whole point!"); return
    if isinstance(guess, str) and isinstance(actual, str):
        guess, actual = guess.strip().lower(), actual.strip().lower()
    if isinstance(actual, (int, float)) and isinstance(guess, (int, float)):
        ok = abs(guess - actual) <= tol * max(1.0, abs(actual)) if tol else guess == actual
    else:
        ok = tuple(guess) == tuple(actual) if isinstance(guess, (tuple, list)) else guess == actual
    PRED[name] = ok
    print(f"{'✅' if ok else '❌'} {name}: you guessed {guess}, actual {actual}")

def intuition_score():
    if not PRED: print("No predictions recorded yet."); return
    n = sum(PRED.values())
    print(f"🔮 Intuition score: {n}/{len(PRED)} predictions right")
    for k, v in PRED.items(): print(("   ✅ " if v else "   ❌ ") + k)

# ---------- TA checkpoints
_CP = {1: "f394aa7979e5cc5d490493628d72a3847b31a79c38a81db1da7df4ef1cf8be23", 2: "1b6a3443f30a8adc08c12490d5b12cbe9a8590f6c76b73a15cdb3ac880eb7cc4", 3: "b2e78e3a5bb1d60df728818ea576deafd6ba9b428fefe13794e012f226516dad"}
UNLOCKED = set()
def checkpoint(n, word):
    if hashlib.sha256(word.strip().lower().encode()).hexdigest() == _CP[n]:
        UNLOCKED.add(n); print(f"🔓 Checkpoint {n} passed. Nice work!")
    else:
        print("🔒 Not the right word. Explain your answer to a TA to get it.")
print("Setup done ✅")

In [ ]:
# Load CIFAR-10 and keep the 4 classes from Lab 4
NAMES = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
CLASSES = ["airplane", "automobile", "cat", "ship"]
ids = [NAMES.index(c) for c in CLASSES]

def pick(ds, per_class, offset=0):
    y = np.array(ds.targets); X, Y = [], []
    for new, old in enumerate(ids):
        idx = np.where(y == old)[0][offset:offset + per_class]
        X.append(ds.data[idx]); Y.append(np.full(len(idx), new))
    X, Y = np.concatenate(X), np.concatenate(Y)
    p = np.random.RandomState(1).permutation(len(Y))
    return X[p], Y[p]

tr = torchvision.datasets.CIFAR10("data", train=True, download=True)
te = torchvision.datasets.CIFAR10("data", train=False, download=True)
Xtr_np, ytr_np = pick(tr, 2000)            # 8,000 training images
Xva_np, yva_np = pick(tr, 500, offset=2000) # 2,000 validation images
Xte_np, yte_np = pick(te, 1000)            # 4,000 test images: touch ONCE, at the very end

MEAN = Xtr_np.reshape(-1, 3).mean(0) / 255; STD = Xtr_np.reshape(-1, 3).std(0) / 255
def to_tensor(X):   # uint8 (N,32,32,3) -> normalized float (N,3,32,32)
    x = torch.tensor(X, dtype=torch.float32).permute(0, 3, 1, 2) / 255
    return ((x - torch.tensor(MEAN, dtype=torch.float32).view(1, 3, 1, 1)) / torch.tensor(STD, dtype=torch.float32).view(1, 3, 1, 1))
Xtr, ytr = to_tensor(Xtr_np).to(device), torch.tensor(ytr_np).to(device)
Xva, yva = to_tensor(Xva_np).to(device), torch.tensor(yva_np).to(device)
Xte, yte = to_tensor(Xte_np).to(device), torch.tensor(yte_np).to(device)
print("train", tuple(Xtr.shape), "· val", tuple(Xva.shape), "· test", tuple(Xte.shape))

fig, ax = plt.subplots(1, 8, figsize=(12, 1.8))
for a, i in zip(ax, range(8)):
    a.imshow(Xtr_np[i]); a.set_title(CLASSES[ytr_np[i]], fontsize=9); a.axis("off")
plt.show()

✏️ **Write down your Lab 4 result** (HOG + SVM validation accuracy on these same 4 classes). If you don't have it, use 70%.

In [ ]:
LAB4_ACCURACY = ___   # ✍️ e.g. 0.72

---
## Mission 1 · Build a CNN (25 min)

Here's the CNN we'll train, the same recipe as the lecture: **(conv → ReLU → pool) × 3 → flatten → fully connected**.
All convolutions are 3×3 with `padding=1`, and all pools are 2×2 max pools.

### 🔮 1a · Predict every shape before you build it
Input: one CIFAR image, **3 × 32 × 32** (channels × height × width). Use `output = (N − F + 2P)/S + 1` from the lecture.

In [ ]:
# ✍️ Fill in your predictions as (channels, height, width) BEFORE running the next cells
MY_GUESSES = {
    "after conv1 (16 filters)": (___),
    "after pool1":              (___),
    "after conv2 (32 filters)": (___),
    "after pool2":              (___),
    "after conv3 (64 filters)": (___),
    "after pool3":              (___),
}
MY_FLATTEN_SIZE = ___            # how many numbers go into the final Linear layer?
MY_PARAMS_CONV2 = ___            # weights + biases in conv2  (hint: 32 filters, each 3×3×16, plus 1 bias each)
MY_TOTAL_PARAMS = ___           # total learnable numbers in the whole network (your best guess!)

Now build it. Two blanks: how many input channels does `conv2` get, and how many numbers come out of the flatten?

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(___, 32, 3, padding=1)     # ✍️ input channels = output channels of conv1
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(___, n_classes)        # ✍️ flatten size
    def forward(self, x, trace=False):
        shapes = {}
        x = F.relu(self.conv1(x)); shapes["after conv1 (16 filters)"] = tuple(x.shape[1:])
        x = self.pool(x);          shapes["after pool1"] = tuple(x.shape[1:])
        x = F.relu(self.conv2(x)); shapes["after conv2 (32 filters)"] = tuple(x.shape[1:])
        x = self.pool(x);          shapes["after pool2"] = tuple(x.shape[1:])
        x = F.relu(self.conv3(x)); shapes["after conv3 (64 filters)"] = tuple(x.shape[1:])
        x = self.pool(x);          shapes["after pool3"] = tuple(x.shape[1:])
        x = x.flatten(1)
        out = self.fc(x)
        return (out, shapes) if trace else out

model = SmallCNN().to(device)
_, real = model(Xtr[:1], trace=True)
for k in MY_GUESSES: check(k, MY_GUESSES[k], real[k])
check("flatten size", MY_FLATTEN_SIZE, real["after pool3"][0] * real["after pool3"][1] * real["after pool3"][2])
check("conv2 parameters", MY_PARAMS_CONV2, sum(p.numel() for p in model.conv2.parameters()))
check("total parameters (±10%)", MY_TOTAL_PARAMS, sum(p.numel() for p in model.parameters()), tol=0.10)

🤔 **Discuss with your partner:** the lecture's fully connected network for a phone photo needed **trillions** of weights. This CNN has fewer than 30,000. Which **two ideas** from the lecture make that possible?

Answer: _______________

### 🧩 1b · Write the training step (Parsons puzzle)
Every training loop in the world runs these **five lines**, but they're scrambled below. Put them in the right order inside `train_step`.

```python
opt.step()                     # update the weights
loss = loss_fn(scores, y)      # how wrong are we?
loss.backward()                # backprop: compute every gradient
opt.zero_grad()                # erase the gradients from the previous step
scores = model(x)              # forward pass
```

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def train_step(model, opt, x, y):
    # ✍️ paste the five lines here, in the right order
    ___
    return loss.item()

# --- automatic check: one step on a tiny problem must reduce the loss
_m = SmallCNN().to(device); _o = torch.optim.SGD(_m.parameters(), lr=0.05)
_l0 = loss_fn(_m(Xtr[:16]), ytr[:16]).item()
for _ in range(5): train_step(_m, _o, Xtr[:16], ytr[:16])
_l1 = loss_fn(_m(Xtr[:16]), ytr[:16]).item()
print("✅ train_step works: loss went down" if _l1 < _l0 else "❌ The loss didn't go down. Check the order of your five lines.", f"({_l0:.3f} → {_l1:.3f})")

🤔 **Why must `zero_grad()` come before `backward()`?** (Hint: PyTorch *adds* new gradients to old ones.)

Answer: _______________

### 🔮 1c · Two sanity checks that professionals always run
**Check 1, the starting loss.** An untrained network guesses about equally among the 4 classes. The cross-entropy loss is −ln(probability of the correct class). What should the starting loss be?

In [ ]:
MY_START_LOSS = ___      # ✍️ a number (calculator allowed)

model = SmallCNN().to(device)
with torch.no_grad():
    start = loss_fn(model(Xtr[:512]), ytr[:512]).item()
check("starting loss (±5%)", MY_START_LOSS, round(start, 3), tol=0.05)

**Check 2, overfit 20 images.** If the network can't even *memorize* 20 images, something is broken, and no tuning will save you.

In [ ]:
model = SmallCNN().to(device); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
x20, y20 = Xtr[:20], ytr[:20]
hist = [train_step(model, opt, x20, y20) for _ in range(150)]
plt.figure(figsize=(5, 2.6)); plt.plot(hist); plt.xlabel("step"); plt.ylabel("training loss"); plt.title("Memorizing 20 images"); plt.show()
print("✅ Loss reached", round(hist[-1], 4), "so the pipeline can learn." if hist[-1] < 0.05 else "Hmm, not near 0. Ask a TA.")

### 🔮 1d · Train for real
Below is the full training loop. It uses **your** `train_step`, plots both curves after every epoch, and keeps the best model by validation accuracy.

**Before you run it:** will this tiny CNN beat your Lab 4 HOG + SVM score? By how much?

In [ ]:
MY_CNN_ACCURACY = ___      # ✍️ your guess for the best validation accuracy, e.g. 0.75

In [ ]:
def augment(x):
    """Random horizontal flip + random crop (pad 4, crop back to 32): cheap data augmentation on the GPU."""
    flip = torch.rand(x.size(0), device=x.device) < 0.5
    x = torch.where(flip[:, None, None, None], x.flip(3), x)
    xp = F.pad(x, (4, 4, 4, 4), mode="reflect")
    i, j = np.random.randint(0, 9, 2)
    return xp[:, :, i:i + 32, j:j + 32]

@torch.no_grad()
def evaluate(model, X, y, bs=1000):
    model.eval(); loss, correct = 0.0, 0
    for k in range(0, len(X), bs):
        s = model(X[k:k + bs]); loss += loss_fn(s, y[k:k + bs]).item() * len(s); correct += (s.argmax(1) == y[k:k + bs]).sum().item()
    return loss / len(X), correct / len(X)

def train(model, epochs=8, lr=1e-3, batch=128, opt_name="adam", weight_decay=0.0, use_augment=False,
          X=None, y=None, time_budget=None, title="", plot=True, quiet=False):
    X = Xtr if X is None else X; y = ytr if y is None else y
    if opt_name == "adam": opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else: opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    H = {"train": [], "val": [], "val_acc": []}; best = (-1, None); t0 = time.time()
    for ep in range(epochs):
        model.train(); p = torch.randperm(len(X), device=X.device); tot = 0.0
        for k in range(0, len(X), batch):
            xb, yb = X[p[k:k + batch]], y[p[k:k + batch]]
            if use_augment: xb = augment(xb)
            tot += train_step(model, opt, xb, yb) * len(xb)
        vl, va = evaluate(model, Xva, yva)
        H["train"].append(tot / len(X)); H["val"].append(vl); H["val_acc"].append(va)
        if va > best[0]: best = (va, copy.deepcopy(model.state_dict()))
        if plot:
            clear_output(wait=True)
            fig, ax = plt.subplots(1, 2, figsize=(10, 3))
            e = range(1, len(H["train"]) + 1)
            ax[0].plot(e, H["train"], "o-", label="training loss"); ax[0].plot(e, H["val"], "s-", label="validation loss")
            ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title(title or "Loss")
            ax[1].plot(e, H["val_acc"], "o-", color="green"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("validation accuracy")
            ax[1].set_title(f"best so far: {best[0]:.1%}"); plt.tight_layout(); plt.show()
        elif not quiet:
            print(f"epoch {ep + 1}: train loss {H['train'][-1]:.3f} · val loss {vl:.3f} · val acc {va:.1%}")
        if time_budget and time.time() - t0 > time_budget:
            print(f"⏱ time budget reached after {ep + 1} epochs"); break
        if not math.isfinite(H["train"][-1]):
            print("💥 The loss became NaN (not a number). Training stopped."); break
    if best[1] is not None: model.load_state_dict(best[1])
    print(f"Done in {time.time() - t0:.0f}s · best validation accuracy {max(H['val_acc']):.1%}")
    return H

EPOCHS = 8   # use 4 if you're on CPU
model = SmallCNN().to(device)
H1 = train(model, epochs=EPOCHS, title="SmallCNN on 8,000 images")
check("CNN accuracy (±5 points)", MY_CNN_ACCURACY, round(max(H1["val_acc"]), 3), tol=0.05)
print(f"\nLab 4 HOG + SVM: {LAB4_ACCURACY:.0%}   →   CNN: {max(H1['val_acc']):.0%}")

✏️ **Write your CNN accuracy on the board next to your Lab 4 number.**

🤔 **Discuss:** The CNN never saw a HOG feature. What did it learn *instead*, and where in the network? (Mission 3 lets you see it.)

Answer: _______________

### 🎤 TA checkpoint 1
Call a TA. Explain: **(1)** why conv1's output is 16 × 32 × 32 and the flatten size is 1,024, and **(2)** why the starting loss was about 1.39.

In [ ]:
checkpoint(1, "___")    # ✍️ the code word your TA gives you

---
## Mission 2 · Training doctor 🩺 (25 min)

Four teams trained the SmallCNN and each run went wrong. **The bugs are hidden.** You can only see the evidence, just like in a real job.

Your tools:
- `mystery(k)` replays run *k* and shows its curves.
- `peek_settings(k)` shows the hyperparameters that team used.
- `peek_data(k)` shows 8 training images with the labels the network was given.
- `peek_model(k)` prints the network the team used.

For each run: look at the curves first, form a hypothesis, then use **one** tool to confirm it. Fill in the table as you go.

In [ ]:
_MYSTERY = json.loads(base64.b64decode(
    "eyIxIjogeyJsciI6IDMuMCwgIm9wdCI6ICJzZ2QiLCAiZXBvY2hzIjogNiwgIm5fdHJhaW4iOiA4MDAwfSwgIjIiOiB7ImxyIjogMC4wMDEsICJvcHQiOiAiYWRhbSIsICJlcG9jaHMiOiA2LCAibl90cmFpbiI6IDgwMDAsICJzaHVmZmxlIjogdHJ1ZX0sICIzIjogeyJsciI6IDAuMDAxLCAib3B0IjogImFkYW0iLCAiZXBvY2hzIjogMjUsICJuX3RyYWluIjogMjAwfSwgIjQiOiB7ImxyIjogMC4wMDEsICJvcHQiOiAiYWRhbSIsICJlcG9jaHMiOiAxMCwgIm5fdHJhaW4iOiA4MDAwLCAidGlueSI6IHRydWV9fQ=="))

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__(); self.conv = nn.Conv2d(3, 2, 3, padding=1); self.fc = nn.Linear(2 * 4 * 4, 4)
    def forward(self, x):
        return self.fc(F.max_pool2d(F.relu(self.conv(x)), 8).flatten(1))

def _setup(k):
    c = _MYSTERY[str(k)]; g = torch.Generator().manual_seed(k)
    n = c["n_train"]; X, y = Xtr[:n], ytr[:n].clone()
    if c.get("shuffle"): y = y[torch.randperm(n, generator=g).to(y.device)]
    torch.manual_seed(k)
    m = (TinyCNN() if c.get("tiny") else SmallCNN()).to(device)
    return c, m, X, y

def mystery(k):
    c, m, X, y = _setup(k)
    return train(m, epochs=c["epochs"], lr=c["lr"], opt_name=c["opt"], X=X, y=y, title=f"Mystery run {k}")
def peek_settings(k):
    c = _MYSTERY[str(k)]
    print(f"Run {k}: optimizer={c['opt']}, learning rate={c['lr']}, epochs={c['epochs']}, training images={c['n_train']}, batch=128")
def peek_data(k):
    c, m, X, y = _setup(k)
    fig, ax = plt.subplots(1, 8, figsize=(12, 1.8))
    for a, i in zip(ax, range(8)):
        a.imshow(Xtr_np[i]); a.set_title("label: " + CLASSES[y[i].item()], fontsize=9); a.axis("off")
    plt.suptitle(f"Run {k}: what the network was told"); plt.show()
def peek_model(k):
    c, m, X, y = _setup(k); print(m); print("parameters:", sum(p.numel() for p in m.parameters()))
print("🩺 Ready. Start with mystery(1).")

In [ ]:
mystery(1)

In [ ]:
mystery(2)

In [ ]:
mystery(3)

In [ ]:
mystery(4)

🔍 **Investigate.** Use the peek tools in this cell, as many times as you like.

In [ ]:
# ✍️ add as many peeks as you need, e.g. peek_settings(1), peek_data(2), peek_model(4)
peek_settings(1)

📋 **Your diagnosis table** (double-click to edit):

| Run | What the curves show | Your diagnosis | Evidence (which tool proved it?) | Fix |
|---|---|---|---|---|
| 1 | _______________ | _______________ | _______________ | _______________ |
| 2 | _______________ | _______________ | _______________ | _______________ |
| 3 | _______________ | _______________ | _______________ | _______________ |
| 4 | _______________ | _______________ | _______________ | _______________ |

### 🔧 Now fix two of them
**Fix run 1:** retrain the same model with a learning rate you choose. 🔮 First predict: what is the lowest SGD learning rate that still reaches ≥ 70% in 6 epochs? Try a few.

In [ ]:
c, m, X, y = _setup(1)
H = train(m, epochs=6, lr=___, opt_name="sgd", title="Run 1, fixed")     # ✍️ your learning rate

**Fix run 3 (overfitting) without adding data.** You may only use `use_augment=True` and/or `weight_decay=...`.

🔮 Which helps more on 200 images, augmentation or weight decay? Guess first.

In [ ]:
MY_BETTER_FIX = "___"     # ✍️ "augment" or "weight_decay"

results = {}
for name, kw in {"nothing": {}, "augment": {"use_augment": True}, "weight_decay": {"weight_decay": 1e-3}}.items():
    c, m, X, y = _setup(3)
    h = train(m, epochs=25, lr=1e-3, X=X, y=y, plot=False, quiet=True, **kw)
    results[name] = max(h["val_acc"])
print({k: f"{v:.1%}" for k, v in results.items()})
check("better fix for 200 images", MY_BETTER_FIX, max(["augment", "weight_decay"], key=results.get))

🤔 **Discuss:** Run 2 never improved, but the code had no error. Why is that scarier than run 1's crash? What check from Mission 1 would have caught it **before** training?

Answer: _______________

### 🎤 TA checkpoint 2
Show a TA your table. For each run, name the diagnosis **and the evidence that proves it**.

In [ ]:
checkpoint(2, "___")

---
## Mission 3 · See what it sees 👀 (15 min)

Now meet a **real** network: ResNet-18, trained on 1.2 million ImageNet photos. Let's open it up.

In [ ]:
weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1
resnet = torchvision.models.resnet18(weights=weights).to(device).eval()
IMNET_CLASSES = weights.meta["categories"]

f = resnet.conv1.weight.detach().cpu()          # 64 filters, each 3 × 7 × 7
f = (f - f.min()) / (f.max() - f.min())
fig, ax = plt.subplots(4, 16, figsize=(12, 3.2))
for a, k in zip(ax.ravel(), range(64)):
    a.imshow(f[k].permute(1, 2, 0)); a.axis("off")
plt.suptitle("ResNet-18, layer 1: the 64 filters it learned by itself"); plt.show()

🤔 **Discuss:** Sort the 64 filters into 2–3 groups. Which group looks like the gradient detectors **you designed by hand in Module 4 (HOG)**? What does the other group detect?

Answer: _______________

### 📸 Your photo
Take a photo with your webcam (or upload one). The helper below works in Colab; if the camera doesn't work, it lets you upload a file instead.

In [ ]:
from PIL import Image
import io
def _colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

def record_frames(n=1, interval_ms=300, countdown=3, label=""):
    """Opens the webcam in Colab. Click the button; after a countdown it records n frames. Returns a list of PIL images."""
    from IPython.display import Javascript
    from google.colab.output import eval_js
    display(Javascript("""
    async function recordFrames(n, interval, countdown, label) {
      const div = document.createElement('div');
      const info = document.createElement('div'); info.style.fontSize = '18px'; info.style.margin = '6px';
      info.textContent = label + ' Click the button when you are ready.';
      const btn = document.createElement('button'); btn.textContent = '🎥 Record'; btn.style.fontSize = '16px';
      const video = document.createElement('video'); video.style.display = 'block'; video.style.maxWidth = '360px';
      div.appendChild(info); div.appendChild(btn); div.appendChild(video); document.body.appendChild(div);
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      video.srcObject = stream; await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => btn.onclick = resolve);
      for (let c = countdown; c > 0; c--) { info.textContent = label + ' Starting in ' + c + '…'; await new Promise(r => setTimeout(r, 1000)); }
      const canvas = document.createElement('canvas'); canvas.width = video.videoWidth; canvas.height = video.videoHeight;
      const frames = [];
      for (let i = 0; i < n; i++) {
        info.textContent = label + ' Recording ' + (i + 1) + ' / ' + n + ': move the object around!';
        canvas.getContext('2d').drawImage(video, 0, 0); frames.push(canvas.toDataURL('image/jpeg', 0.85));
        await new Promise(r => setTimeout(r, interval));
      }
      stream.getVideoTracks()[0].stop(); div.remove();
      return frames;
    }"""))
    urls = eval_js(f"recordFrames({n}, {interval_ms}, {countdown}, {json.dumps(label)})")
    return [Image.open(io.BytesIO(base64.b64decode(u.split(",")[1]))).convert("RGB") for u in urls]

def upload_images():
    from google.colab import files
    up = files.upload()
    return {name: Image.open(io.BytesIO(b)).convert("RGB") for name, b in up.items()}

def get_photo():
    try:
        return record_frames(1, label="📸 Your photo.")[0]
    except Exception as e:
        print("Camera not available (", e, "). Upload a photo instead.")
        try:
            return next(iter(upload_images().values()))
        except Exception as e2:
            print("No upload either; using a sample image.", e2)
            from skimage import data as skdata
            return Image.fromarray(skdata.chelsea())

my_photo = get_photo()
plt.figure(figsize=(3, 3)); plt.imshow(my_photo); plt.axis("off"); plt.show()

🔮 **Predict:** ImageNet has 1,000 classes but **no "person" class**. What will ResNet-18 say your photo is?

In [ ]:
MY_RESNET_GUESS = "___"   # ✍️

IM_MEAN, IM_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
import torchvision.transforms as T
prep = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(IM_MEAN, IM_STD)])

with torch.no_grad():
    p = resnet(prep(my_photo)[None].to(device)).softmax(1)[0]
top = p.topk(5)
for prob, i in zip(top.values, top.indices):
    print(f"{prob:6.1%}  {IMNET_CLASSES[i]}")

Now look **inside**: which feature maps react most strongly to your photo, early in the network vs. deep inside it?

In [ ]:
acts = {}
hooks = [getattr(resnet, n).register_forward_hook(lambda m, i, o, n=n: acts.__setitem__(n, o.detach().cpu()))
         for n in ["layer1", "layer4"]]
with torch.no_grad(): resnet(prep(my_photo)[None].to(device))
for h in hooks: h.remove()

for name in ["layer1", "layer4"]:
    a = acts[name][0]                                   # channels × H × W
    top_ch = a.mean((1, 2)).topk(6).indices
    fig, ax = plt.subplots(1, 7, figsize=(13, 2.2))
    ax[0].imshow(my_photo.resize((224, 224))); ax[0].set_title("input", fontsize=9)
    for j, c in enumerate(top_ch):
        ax[j + 1].imshow(a[c], cmap="viridis"); ax[j + 1].set_title(f"{name} ch {c.item()}", fontsize=9)
    for x in ax: x.axis("off")
    plt.suptitle(f"{name}: maps are {tuple(a.shape[1:])}, {a.shape[0]} channels"); plt.show()

🤔 **Discuss:** layer 1 maps are 56 × 56 and still look like your photo. Layer 4 maps are only 7 × 7. What changed, and why is that **good** for classification but a **problem** for detection (Part 4 of the lecture)?

Answer: _______________

---
## Mission 4 · Teach it your objects 🎓 (25 min)

You'll build a **3-object recognizer** from about 45 photos you record yourselves, using transfer learning.

1. Pick **3 objects** you have right now (for example a phone, a water bottle and a pen, or 3 hand signs).
2. For each object, record **15 frames**. **Move it around**: turn it, change the distance, tilt it. Variety is your free data augmentation.

No camera? Upload photos named `classname_1.jpg`, `classname_2.jpg`, … or use the CIFAR fallback.

In [ ]:
MY_OBJECTS = ["___", "___", "___"]     # ✍️ your 3 object names
FRAMES = 15

In [ ]:
def collect():
    assert "___" not in MY_OBJECTS, "Write your 3 object names in the cell above first."
    imgs, labels = [], []
    try:
        for c, name in enumerate(MY_OBJECTS):
            frames = record_frames(FRAMES, label=f"Object {c + 1}/3: show the {name.upper()}.")
            imgs += frames; labels += [c] * len(frames)
        return imgs, labels
    except Exception as e:
        print("Camera not available:", e)
    try:
        print(f"Upload photos named like {MY_OBJECTS[0]}_1.jpg ...")
        up = upload_images()
        for fname, im in up.items():
            name = fname.split("_")[0].lower()
            if name in MY_OBJECTS: imgs.append(im); labels.append(MY_OBJECTS.index(name))
        if len(set(labels)) == 3: return imgs, labels
    except Exception as e:
        print("No upload:", e)
    print("➡️ Fallback: 15 CIFAR images each of airplane / automobile / ship.")
    MY_OBJECTS[:] = ["airplane", "automobile", "ship"]
    for c, name in enumerate(MY_OBJECTS):
        idx = np.where(ytr_np == CLASSES.index(name))[0][:FRAMES]
        imgs += [Image.fromarray(Xtr_np[i]) for i in idx]; labels += [c] * len(idx)
    return imgs, labels

imgs, labels = collect()
labels = np.array(labels)
fig, ax = plt.subplots(1, 9, figsize=(13, 1.8))
for a, i in zip(ax, np.linspace(0, len(imgs) - 1, 9).astype(int)):
    a.imshow(imgs[i]); a.set_title(MY_OBJECTS[labels[i]], fontsize=9); a.axis("off")
plt.show(); print(len(imgs), "images")

Split: the **last 5 frames of each object** go into validation. Frames recorded one after another look almost identical, so a random split would leak (Module 4!).

In [ ]:
val_mask = np.zeros(len(labels), bool)
for c in range(3):
    idx = np.where(labels == c)[0]; val_mask[idx[-5:]] = True
tr_idx, va_idx = np.where(~val_mask)[0], np.where(val_mask)[0]
print("train:", len(tr_idx), "· validation:", len(va_idx))

### 🔮 4a · From scratch vs. pretrained
We'll compare two models trained on **the same ~30 images**:
- **A · From scratch:** your SmallCNN from Mission 1, starting with random weights (images resized to 32 × 32).
- **B · Transfer learning:** ResNet-18 frozen as a feature extractor, plus a new 3-class head.

In [ ]:
MY_SCRATCH_ACC  = ___    # ✍️ guess the validation accuracy of A (random guessing = 0.33)
MY_TRANSFER_ACC = ___    # ✍️ guess for B

In [ ]:
# ---- A: from scratch
small = lambda im: np.array(im.resize((32, 32)))
Xs = to_tensor(np.stack([small(im) for im in imgs])).to(device); ys = torch.tensor(labels).to(device)
torch.manual_seed(0); scratch = SmallCNN(n_classes=3).to(device)
opt = torch.optim.Adam(scratch.parameters(), lr=1e-3)
for ep in range(60):
    scratch.train(); train_step(scratch, opt, augment(Xs[tr_idx]), ys[tr_idx])
scratch.eval()
with torch.no_grad(): acc_A = (scratch(Xs[va_idx]).argmax(1) == ys[va_idx]).float().mean().item()

# ---- B: transfer learning (frozen backbone = feature extractor)
backbone = copy.deepcopy(resnet); backbone.fc = nn.Identity(); backbone.eval()
@torch.no_grad()
def features(images, transform=prep, model=None):
    model = backbone if model is None else model
    return torch.cat([model(transform(im)[None].to(device)) for im in images])
feats = features(imgs)                              # one 512-number vector per image
head = nn.Linear(512, 3).to(device); opt = torch.optim.Adam(head.parameters(), lr=1e-3)
for ep in range(200):
    train_step(head, opt, feats[tr_idx], ys[tr_idx])
with torch.no_grad(): acc_B = (head(feats[va_idx]).argmax(1) == ys[va_idx]).float().mean().item()

print(f"A · from scratch:      {acc_A:.0%}\nB · transfer learning: {acc_B:.0%}")
check("from-scratch accuracy (±15 points)", MY_SCRATCH_ACC, round(acc_A, 2), tol=0.15)
check("transfer accuracy (±10 points)", MY_TRANSFER_ACC, round(acc_B, 2), tol=0.10)

### 🐛 4b · The silent bugs
The lecture warned about three details that **silently** break transfer learning. Let's commit two of them on purpose and measure the damage.

🔮 Which hurts more: **forgetting to normalize**, or **running the backbone in training mode** with one image at a time?

In [ ]:
MY_WORSE_BUG = "___"      # ✍️ "no normalize" or "train mode"

no_norm = T.Compose([T.Resize((224, 224)), T.ToTensor()])          # bug 1: forgot T.Normalize
bad_model = copy.deepcopy(backbone).train()                          # bug 2: forgot model.eval()
with torch.no_grad():
    acc_nonorm = (head(features([imgs[i] for i in va_idx], transform=no_norm)).argmax(1) == ys[va_idx]).float().mean().item()
    acc_train  = (head(features([imgs[i] for i in va_idx], model=bad_model)).argmax(1) == ys[va_idx]).float().mean().item()
print(f"correct pipeline:     {acc_B:.0%}\nforgot normalization: {acc_nonorm:.0%}\nforgot model.eval():  {acc_train:.0%}")
check("worse bug", MY_WORSE_BUG, "no normalize" if acc_nonorm < acc_train else "train mode")

🤔 **Discuss:** Neither bug raised an error. Explain, in one sentence each, what goes wrong inside the network.

Answer: _______________

### 🎬 4c · Live demo
Hold one of your objects (or a *new* one!) in front of the camera.

In [ ]:
def live_test():
    im = get_photo()
    with torch.no_grad(): p = head(features([im])).softmax(1)[0]
    plt.figure(figsize=(3, 3)); plt.imshow(im); plt.axis("off")
    plt.title(" · ".join(f"{n} {q:.0%}" for n, q in zip(MY_OBJECTS, p.tolist())), fontsize=9); plt.show()
live_test()

🤔 **Try to fool it:** show it something that is **none** of your 3 objects. What does it say, and how confident is it? Why can't a 3-class softmax say "I don't know"?

Answer: _______________

### 🎤 TA checkpoint 3
Do the live demo for a TA. Explain why B beats A with only about 30 training images, and what each silent bug did.

In [ ]:
checkpoint(3, "___")

---
## Mission 5 · YOLO hunt 🎯 (15 min)

A detector pretrained on COCO (80 classes). Your missions:
1. 🔮 Predict the size of YOLO's raw output.
2. Find **one photo it gets right**, **one object it misses** and **one false alarm**.
3. Play with the two knobs from the lecture: the **confidence threshold** and the **NMS IoU threshold**.

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO
yolo = YOLO("yolov8n.pt")     # the smallest model: about 3 million parameters

**🔮 Predict the raw output.** YOLOv8 looks at a 640 × 640 image on **three grids** with strides 8, 16 and 32, and every grid cell makes one prediction.
How many candidate boxes is that per image? Each candidate has 4 box numbers + one score for each of the 80 COCO classes. How many numbers per candidate?

*(YOLOv8 is "anchor-free": instead of adjusting 3 anchor templates per cell, each cell predicts its box directly. Same grid idea, simpler head.)*

In [ ]:
MY_CANDIDATES = ___     # ✍️ (640/8)² + (640/16)² + (640/32)²
MY_PER_CANDIDATE = ___    # ✍️ 4 + 80

yolo.model.eval()
with torch.no_grad():
    raw = yolo.model(torch.zeros(1, 3, 640, 640).to(next(yolo.model.parameters()).device))
raw = raw[0] if isinstance(raw, (tuple, list)) else raw
print("raw output shape:", tuple(raw.shape), "→ (batch, numbers per candidate, candidates)")
check("candidates per image", MY_CANDIDATES, raw.shape[2])
check("numbers per candidate", MY_PER_CANDIDATE, raw.shape[1])

In [ ]:
def detect(im, conf=0.25, iou=0.7):
    r = yolo(im, conf=conf, iou=iou, verbose=False)[0]          # a PIL image (RGB)
    plt.figure(figsize=(6, 4.5)); plt.imshow(r.plot()[..., ::-1]); plt.axis("off")
    plt.title(f"conf ≥ {conf}, NMS IoU {iou}: {len(r.boxes)} boxes"); plt.show()
    return r

scene = get_photo()          # point the camera at a busy desk, the room, the window...
r = detect(scene)

🔮 **Knob 1, confidence.** How many boxes will survive at `conf=0.05`, and at `conf=0.8`? Write your guesses, then run.

In [ ]:
MY_BOXES_LOW, MY_BOXES_HIGH = ___, ___    # ✍️
n_low = len(detect(scene, conf=0.05).boxes); n_high = len(detect(scene, conf=0.8).boxes)
check("boxes at conf 0.05 (±50%)", MY_BOXES_LOW, n_low, tol=0.5); check("boxes at conf 0.8 (±1)", MY_BOXES_HIGH, n_high, tol=1.0)

🔮 **Knob 2, NMS.** What do you expect to see if the NMS IoU threshold is set very high (`iou=0.95`)? Very low (`iou=0.1`)?

In [ ]:
_ = detect(scene, conf=0.1, iou=0.95)
_ = detect(scene, conf=0.1, iou=0.1)

🤔 **Discuss:** Explain what you saw with each NMS setting. For a crowd of people standing close together, which setting would you choose, and what is the risk?

Answer: _______________

🏆 **Class challenge, "Fool YOLO":** take a photo that makes YOLO produce the most confident **wrong** box. Post it in the class channel with the confidence. Winners are announced at the next lecture.

**Your hunt log:**
- ✅ Correct detection: _______________
- ❌ A missed object, and why you think it was missed (size? angle? not in COCO?): _______________
- 👻 A false alarm, and why: _______________

---
## 🏁 Speed-run leaderboard (optional, 10 min)
Beat the class on the **4-class CIFAR test set**. Rules:
- **2 minutes** of training (`time_budget=120`).
- Change anything: channels, an extra conv layer, dropout, augmentation, learning rate, weight decay.
- Test **once**, at the very end. If you test, change something and test again, you're tuning on the test set (Module 4: never!).

In [ ]:
class MyCNN(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        # ✍️ your design here (start from SmallCNN and improve it)
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1); self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1); self.pool = nn.MaxPool2d(2)
        self.drop = nn.Dropout(0.3); self.fc = nn.Linear(128 * 4 * 4, n_classes)
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))); x = self.pool(F.relu(self.conv2(x))); x = self.pool(F.relu(self.conv3(x)))
        return self.fc(self.drop(x.flatten(1)))

best_model = MyCNN().to(device)
H = train(best_model, epochs=100, lr=1e-3, use_augment=True, weight_decay=1e-4, time_budget=120, title="Speed run")

In [ ]:
# 🔒 Run ONCE, when you're done tuning on validation
test_loss, test_acc = evaluate(best_model, Xte, yte)
print(f"🏁 TEST accuracy: {test_acc:.1%}  ←  post this on the leaderboard (with your validation accuracy: {max(H['val_acc']):.1%})")

---
## 📝 Engineering log (5 min, in your own words, 2–3 sentences each)

**1.** Your model trains but its accuracy is bad. Write down, **in order**, the first three things you'd check, and why.

_______________

**2.** A friend has 300 photos of 5 kinds of plant disease. Should they train a CNN from scratch? What exactly would you tell them to do?

_______________

**3.** Which moment today surprised you most, and why?

_______________

In [ ]:
intuition_score()
print("Checkpoints passed:", sorted(UNLOCKED), "/ [1, 2, 3]")